# 03 · Within-group comparison of Life:Me and Death:Me (Results 3.1, Figure 2, Table S5)

For each group: **paired-samples t-test** (t, df, p, d_z = mean difference / SD of the differences, 95% CI of the
difference) and the Wilcoxon signed-rank test. The **Group × Condition mixed ANOVA** is computed exactly from
difference scores: the interaction is a one-way ANOVA on (Life:Me − Death:Me), the group main effect a one-way ANOVA
on the mean of the two conditions, and the condition main effect the intercept of the difference-score model
(sum-to-zero coding). The analysis is run for RTs of all trials, RTs of correct trials, and error rates.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
import sys
sys.path.insert(0, "..")          # dbiat_analysis.py is in the folder above notebooks/
import dbiat_analysis as A

cfg = A.load_settings()
df = pd.read_csv(A.path(cfg, "derived", "scores.csv"))
pd.set_option("display.width", 250)

## 1. Paired tests per group

In [ ]:
PAIRS = [("RT overall", "life_rt", "death_rt"), ("RT correct", "life_rt_correct", "death_rt_correct"),
         ("Error rate", "life_er", "death_er")]
within = pd.DataFrame([dict(measure=label, group=g, **A.paired_tests(d[a], d[b]))
                       for label, a, b in PAIRS
                       for g, d in [(g, df[df.group == g]) for g in A.GROUPS] + [("All", df)]])
A.write_table(within, cfg, "within_group_paired_tests")
within.round(4)

## 2. Group × Condition mixed ANOVA

In [ ]:
def mixed_anova(a, b):
    d = df.assign(diff=df[a] - df[b], avg=(df[a] + df[b]) / 2)
    inter, grp = A.oneway_anova(d, "diff"), A.oneway_anova(d, "avg")
    fit = smf.ols("diff ~ C(group, Sum)", data=d).fit()
    return dict(condition_F=fit.tvalues["Intercept"] ** 2, condition_df=f"(1, {int(fit.df_resid)})",
                condition_p=fit.pvalues["Intercept"],
                group_F=grp["F"], group_df=f"({grp['df1']}, {grp['df2']})", group_p=grp["p"],
                interaction_F=inter["F"], interaction_df=f"({inter['df1']}, {inter['df2']})",
                interaction_p=inter["p"], interaction_eta2=inter["eta2"])

mixed = pd.DataFrame([dict(measure=label, **mixed_anova(a, b)) for label, a, b in PAIRS])
A.write_table(mixed, cfg, "within_group_mixed_anova")
mixed.round(4)

## 3. Figure 2
Style of the submitted figure. Bars show group means with t-based 95% confidence intervals; the p-value is from the
paired t-test. The figure is drawn at its printed width (180 mm) so that all text is at least 8 pt.

In [ ]:
w = within[within.measure == "RT overall"].set_index("group")
pfmt = lambda p: "p < 0.001" if p < .001 else f"p = {p:.3f}"

def figure2(figsize, fs, name):
    with plt.style.context("default"):
        fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)
        for ax, g in zip(axes, A.GROUPS):
            d = df[df.group == g]
            means = [d.life_rt.mean(), d.death_rt.mean()]
            ci = [stats.t.ppf(.975, len(d) - 1) * d[c].std() / np.sqrt(len(d)) for c in ("life_rt", "death_rt")]
            ax.bar(["Life:Me RT", "Death:Me RT"], means, yerr=ci, capsize=4 * fs, color=["skyblue", "salmon"])
            for i, (m, c) in enumerate(zip(means, ci)):
                ax.text(i, m + c + 3, f"{m:.2f}", ha="center", va="bottom", fontsize=13 * fs)
            r = w.loc[g]
            ax.text(0.5, max(np.add(means, ci)) + 22, f"paired {pfmt(r.p)}{'*' if r.p < .05 else ''}",
                    ha="center", fontsize=13 * fs)
            ax.set_title(g, fontsize=16 * fs)
            ax.set_ylim(750, 990)
            ax.tick_params(axis="x", labelsize=12 * fs)
            ax.tick_params(axis="y", labelsize=max(8.5, 10 * fs))
        axes[0].set_ylabel("Reaction time (RT in msec)", fontsize=14 * fs)
        fig.suptitle("Life:Me vs. Death:Me Reaction Times: Within-Group Comparison", fontsize=17 * fs)
        fig.tight_layout(pad=0.4)
        A.save_fig(fig, cfg, name)
        plt.show()

figure2((180 / 25.4, 3.2), 0.71, "Fig2")
A.finalize_tiff(cfg, "Fig2")